### Day4. 다중 선형 회귀
당뇨병(diabetes) 데이터를 사용해 다중 선형 회귀를 수행합니다.
- 모든 feature는 표준화(z-score 변환) 되어 있어서 평균 0, 분산 1에 가깝습니다.
- age : 환자의 나이
- sex : 성별
- bmi : 체질량지수
- bp : 평균 혈압
- s1 ~ s6 : 6가지 혈액 검사 결과
- target : 1년 후 당뇨병 진행도 지표

In [1]:
# 데이터 확인
import pandas as pd
pd.set_option('display.width', 120)
path = "https://raw.githubusercontent.com/Soyoung-Yoon/bigdata/main/"
df = pd.read_csv(path + "diabetes.csv")
print(df.head(3))

        age       sex       bmi        bp        s1        s2        s3        s4        s5        s6  target
0  0.038076  0.050680  0.061696  0.021872 -0.044223 -0.034821 -0.043401 -0.002592  0.019907 -0.017646   151.0
1 -0.001882 -0.044642 -0.051474 -0.026328 -0.008449 -0.019163  0.074412 -0.039493 -0.068332 -0.092204    75.0
2  0.085299  0.050680  0.044451 -0.005670 -0.045599 -0.034194 -0.032356 -0.002592  0.002861 -0.025930   141.0


다음과 같은 다중선형회귀 모형을 사용한 회귀모델을 만들고 결과를 확인합니다.
- diabetes.csv 데이터를 사용합니다.
- 모델 생성시 상수항(=절편)을 포함하도록 합니다.
- 종속변수 : target
- 독립변수 : target을 제외한 모든 변수

In [6]:
# 4-1) 위의 조건에 맞게 OLS 모델을 생성하고 summary를 출력해 봅니다.
from statsmodels.api import OLS, families
formula = "target ~ " + " + ".join(df.columns[:-1])
model = OLS.from_formula(formula, df).fit()
# model2 = OLS.from_formula(formula, df, family=families.Binomial()).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                 target   R-squared:                       0.518
Model:                            OLS   Adj. R-squared:                  0.507
Method:                 Least Squares   F-statistic:                     46.27
Date:                Sat, 15 Nov 2025   Prob (F-statistic):           3.83e-62
Time:                        12:22:38   Log-Likelihood:                -2386.0
No. Observations:                 442   AIC:                             4794.
Df Residuals:                     431   BIC:                             4839.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    152.1335      2.576     59.061      0.0

In [7]:
#4-2) 위에서 생성한 모델의 결정계수를 반올림하여 소수점 아래 3자리까지 출력합니다.
print(model.rsquared.round(3))

0.518


In [11]:
#4-3) 위에서 생성한 모델의 수정된 결정계수를 반올림하여 소수점 아래 3자리까지 출력합니다.
print(model.rsquared_adj.round(3))

0.507


In [16]:
#4-4) 유의수준 0.05하에서 통계적으로 유의한 독립변수의 개수는 몇 개인가요?
print(sum(model.pvalues[1:] < 0.05))

4


In [24]:
#4-5) 유의수준 0.05하에서 통계적으로 유의한 독립변수만 사용하여 target을 종속변수로 하는 모델을 생성하여
# model2로 저장하고, summary()를 확인합니다.
s = model.pvalues[1:] < 0.05
cols = s[s].index
formula2 = "target ~ " + " + ".join(cols)
model2 = OLS.from_formula(formula2, df).fit()
print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:                 target   R-squared:                       0.487
Model:                            OLS   Adj. R-squared:                  0.482
Method:                 Least Squares   F-statistic:                     103.6
Date:                Sat, 15 Nov 2025   Prob (F-statistic):           5.42e-62
Time:                        13:30:57   Log-Likelihood:                -2399.8
No. Observations:                 442   AIC:                             4810.
Df Residuals:                     437   BIC:                             4830.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    152.1335      2.639     57.648      0.0

In [27]:
#4-6) bmi 변수에 대한 회귀 계수를 구해, 반올림하여 소수점 아래 3자리까지 출력합니다.
print(model2.params['bmi'].round(3))

598.284


In [31]:
#4-7) 영향력이 가장 높은 변수 및 변수의 회귀계수를 구해, 반올림하여 소수점 아래 3자리까지 출력합니다.
# 중간에 공백을 1개 넣어 2개를 한 줄에 출력해 봅니다.
# 출력 예) s1 123.456
parameter = model2.params.idxmax()
value = model2.params[parameter]
print(f"{parameter} {value:.3f}")

bmi 598.284


In [32]:
#4-8) 영향력이 가장 낮은 변수 및 변수의 회귀계수를 구해, 반올림하여 소수점 아래 3자리까지 출력합니다.
# 중간에 공백을 1개 넣어 2개를 한 줄에 출력해 봅니다.
parameter = model2.params.idxmin()
value = model2.params[parameter]
print(f"{parameter} {value:.3f}")

sex -136.758


In [35]:
#4-9) F통계량을 구해, 반올림하여 소수점 아래 3자리까지 출력합니다.
print(model2.fvalue.round(3))

103.618


In [37]:
#4-10) 독립변수 중 가장 높은 p-value를 구해, 반올림하여 소수점 아래 3자리까지 출력합니다.
print(round(model2.pvalues.max(), 3))

0.017


In [39]:
#4-11) 통계적으로 가장 유의한 변수는 무엇인가?
print(model2.pvalues[1:].idxmin())

bmi


In [44]:
#4-12) 통계적으로 가장 유의한 변수의 회귀계수를 구해,
# 반올림하여 소수점 아래 3자리까지 출력합니다.
print(model2.params['bmi'].round(3))

598.284
